# ICICLE vs. NEIMS: example spectra for biggest wins/losses by functional group
#
# For **random-split formula-match**, **scaffold-split formula-match**,
# **global PubChem retrieval** on both random and scaffold splits (~87M
# candidates, no formula restriction), finds functional groups where
# ICICLE's top-1 rate is much higher than NEIMS's (ICICLE-favored) and vice
# versa (NEIMS-favored), shortlists a handful of clean candidate molecules
# per case (winner correct at top-1, loser far from top-1), and plots a
# stacked spectrum for a chosen one: experimental on top (black), ICICLE,
# then NEIMS (both in their standard model colors), fading peaks that
# don't match experimental. Reports rank + number of decoys +
# cosine/entropy similarity for both models.
#
# Self-contained: only loads what this analysis needs, not the full
# similarity/violin-plot machinery of `fig_retrieval_icicle_vs_neims_structural.ipynb`.


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem
import exmol
from tqdm.auto import tqdm
import h5py

from icicle.utils.visualization.eval_plots import model_color
from icicle.utils.visualization.mass_spectra import plot_three_spectra
from icicle.utils.visualization.style import get_palette, save_fig, set_style

set_style("manuscript")
palette = get_palette()

REPO_ROOT = Path("/home/magled/icicle-dev")
EVAL = REPO_ROOT / "results" / "eval"
OUTPUT_DIR = REPO_ROOT / "figures" / "retrieval_win_loss_examples"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ["ICICLE", "NEIMS"]
MODEL_COLORS = {
    m: model_color(m, fallback_index=i) for i, m in enumerate(MODELS)
}
print(MODEL_COLORS)

FORMULA_DIRS = {
    "random": {
        "ICICLE": [
            EVAL / f"final_entropy_random_s{i}_retr" for i in (1, 2, 3)
        ],
        "NEIMS": [EVAL / f"neims_random_s{i}" for i in (1, 2, 3)],
    },
    "scaffold": {
        "ICICLE": [
            EVAL / f"final_entropy_scaffold_s{i}_retr" for i in (1, 2, 3)
        ],
        "NEIMS": [EVAL / f"neims_scaffold_s{i}" for i in (1, 2, 3)],
    },
}
SIM_DIRS = {
    "random": [EVAL / f"final_entropy_random_s{i}_sim" for i in (1, 2, 3)],
    "scaffold": [EVAL / f"final_entropy_scaffold_s{i}_sim" for i in (1, 2, 3)],
}
NEIMS_PRED_FILES = {
    "random": [
        REPO_ROOT
        / "baselines"
        / "neims"
        / "results"
        / "predictions"
        / f"neims_random_noqcxms2_s{i}_test.hdf5"
        for i in (1, 2, 3)
    ],
    "scaffold": [
        REPO_ROOT
        / "baselines"
        / "neims"
        / "results"
        / "predictions"
        / f"neims_scaffold_s{i}_test.hdf5"
        for i in (1, 2, 3)
    ],
}
GLOBAL_RETRIEVAL_DIRS = {
    "random": {
        "ICICLE": REPO_ROOT
        / "results"
        / "pubchem_retrieval_eval_icicle_rerun_260710",
        "NEIMS": REPO_ROOT / "results" / "pubchem_retrieval_eval_neims",
    },
    "scaffold": {
        "ICICLE": REPO_ROOT
        / "results"
        / "pubchem_retrieval_eval_icicle_scaffold_s1",
        "NEIMS": REPO_ROOT
        / "results"
        / "pubchem_retrieval_eval_neims_scaffold_s1",
    },
}

metadata_full = pd.read_csv(
    REPO_ROOT / "data" / "NIST2023_GCMS_main" / "metadata.tsv",
    sep="\t",
    usecols=["mol_id", "mw", "standardized_smiles", "formula", "inchi_key"],
)
metadata_full["inchikey14"] = metadata_full["inchi_key"].astype(str).str[:14]
print(f"metadata.tsv total rows: {len(metadata_full)}")

In [ ]:
def compute_functional_groups(smiles: str) -> frozenset:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return frozenset()
    return frozenset(exmol.get_functional_groups(mol, return_all=True))


def load_icicle_formula_top1(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path / "retrieval_with_formula_results.csv")
    true_rows = raw[~raw["is_decoy"]].rename(columns={"spec": "mol_id"})
    true_rows["top1"] = true_rows["rank_cosine_similarity"] == 1
    return true_rows[["mol_id", "top1"]]


def load_baseline_formula_top1(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path / "retrieval_with_formula_results.csv")
    true_rows = raw[raw["is_correct"]].rename(
        columns={"query_inchikey14": "inchikey14"}
    )
    true_rows["top1"] = true_rows["rank_cosine_similarity"] == 1
    return true_rows.groupby("inchikey14", as_index=False)["top1"].max()


def wide_per_seed(seed_frames: dict, id_col: str, prefix: str) -> tuple:
    wide = None
    for seed, df in seed_frames.items():
        df = df.rename(columns={"top1": f"{prefix}_top1_{seed}"})
        wide = df if wide is None else wide.merge(df, on=id_col, how="outer")
    seed_cols = [f"{prefix}_top1_{seed}" for seed in seed_frames]
    n_present = wide[seed_cols].notna().sum(axis=1)
    wide = wide[n_present == len(seed_frames)].reset_index(drop=True)
    return wide, seed_cols


def build_formula_wide(dirs_by_model: dict, prefix_map: dict) -> dict:
    out = {}
    for model, dirs in dirs_by_model.items():
        seed_frames = {}
        for d in dirs:
            loader = (
                load_icicle_formula_top1
                if model == "ICICLE"
                else load_baseline_formula_top1
            )
            seed_frames[d.name] = loader(d)
        id_col = "mol_id" if model == "ICICLE" else "inchikey14"
        wide, seed_cols = wide_per_seed(seed_frames, id_col, prefix_map[model])
        out[model] = (wide, seed_cols, id_col)
        print(
            f"  {model}: {len(wide)} queries present in all {len(dirs)} seeds"
        )
    return out


def merge_formula_wide(wide_by_model: dict) -> tuple:
    keyed = {}
    for model, (wide, seed_cols, id_col) in wide_by_model.items():
        if id_col == "mol_id":
            wide = wide.merge(
                metadata_full[["mol_id", "inchikey14"]],
                on="mol_id",
                how="left",
            )
        keyed[model] = (wide, seed_cols)

    common_keys = set.intersection(
        *(set(w["inchikey14"]) for w, _ in keyed.values())
    )
    print(
        f"Common inchikey14 across all {len(keyed)} models: {len(common_keys)}"
    )

    merged = metadata_full[metadata_full["inchikey14"].isin(common_keys)][
        ["mol_id", "inchikey14"]
    ].drop_duplicates("inchikey14")
    seed_cols_by_model = {}
    for model, (wide, seed_cols) in keyed.items():
        cols = ["inchikey14"] + seed_cols
        merged = merged.merge(
            wide[wide["inchikey14"].isin(common_keys)][cols],
            on="inchikey14",
            how="inner",
        )
        seed_cols_by_model[model] = seed_cols
    merged = merged.merge(
        metadata_full[["mol_id", "mw", "formula"]], on="mol_id", how="left"
    )
    return merged, seed_cols_by_model


def build_split_merged(split: str) -> pd.DataFrame:
    wide = build_formula_wide(
        FORMULA_DIRS[split], prefix_map={"ICICLE": "icicle", "NEIMS": "neims"}
    )
    merged, seed_cols_by_model = merge_formula_wide(wide)
    for model, seed_cols in seed_cols_by_model.items():
        merged[f"{model}_rate"] = merged[seed_cols].mean(axis=1)

    smiles_map = metadata_full.set_index("mol_id")["standardized_smiles"]
    fg_records = [
        {
            "mol_id": mol_id,
            "functional_groups": compute_functional_groups(smiles_map[mol_id]),
        }
        for mol_id in tqdm(
            merged["mol_id"], desc=f"exmol functional groups ({split})"
        )
    ]
    merged = merged.merge(pd.DataFrame(fg_records), on="mol_id", how="left")
    print(f"[{split}] merged rows: {len(merged)}")
    return merged


def functional_group_summary_n(df, rate_cols: dict, min_count=30):
    cols = ["mol_id", "functional_groups"] + list(rate_cols.values())
    exploded = (
        df[cols]
        .explode("functional_groups")
        .dropna(subset=["functional_groups"])
    )
    agg = {"n": ("mol_id", "count")}
    for model, colname in rate_cols.items():
        agg[f"{model}_rate"] = (colname, "mean")
    summary = exploded.groupby("functional_groups").agg(**agg)
    return summary[summary["n"] >= min_count].round(4)


random_formula_merged = build_split_merged("random")
scaffold_formula_merged = build_split_merged("scaffold")

random_fg_summary = functional_group_summary_n(
    random_formula_merged, {"ICICLE": "ICICLE_rate", "NEIMS": "NEIMS_rate"}
)
scaffold_fg_summary = functional_group_summary_n(
    scaffold_formula_merged, {"ICICLE": "ICICLE_rate", "NEIMS": "NEIMS_rate"}
)

In [ ]:
def build_global_merged(split: str) -> pd.DataFrame:
    per_query = {
        model: pd.read_csv(
            GLOBAL_RETRIEVAL_DIRS[split][model]
            / "retrieval_global_per_query.tsv",
            sep="\t",
        )
        for model in MODELS
    }
    common_ids = set.intersection(
        *(set(df["mol_id"]) for df in per_query.values())
    )
    print(
        f"[global/{split}] Common mol_ids across ICICLE + NEIMS: {len(common_ids)}"
    )

    merged = metadata_full[metadata_full["mol_id"].isin(common_ids)][
        ["mol_id", "mw", "formula", "standardized_smiles"]
    ].copy()
    for model, df in per_query.items():
        sub = df[df["mol_id"].isin(common_ids)][
            ["mol_id", "rank_autofail_cosine", "n_candidates"]
        ]
        sub = sub.rename(
            columns={
                "rank_autofail_cosine": f"{model}_global_rank",
                "n_candidates": f"{model}_global_n_candidates",
            }
        )
        merged = merged.merge(sub, on="mol_id", how="left")
        merged[f"{model}_top1"] = merged[f"{model}_global_rank"] == 1

    merged["functional_groups"] = [
        compute_functional_groups(s)
        for s in tqdm(
            merged["standardized_smiles"],
            desc=f"exmol functional groups (global/{split})",
        )
    ]
    merged["ICICLE_rate"] = merged["ICICLE_top1"].astype(float)
    merged["NEIMS_rate"] = merged["NEIMS_top1"].astype(float)
    print(f"[global/{split}] merged rows: {len(merged)}")
    return merged


global_merged = build_global_merged("random")
global_scaffold_merged = build_global_merged("scaffold")

global_fg_summary = functional_group_summary_n(
    global_merged, {"ICICLE": "ICICLE_rate", "NEIMS": "NEIMS_rate"}
)
global_scaffold_fg_summary = functional_group_summary_n(
    global_scaffold_merged, {"ICICLE": "ICICLE_rate", "NEIMS": "NEIMS_rate"}
)

In [ ]:
def plot_stacked_exp_icicle_neims(**kwargs):
    """plot_three_spectra stacks [ICICLE, NEIMS, exp] top-to-bottom; flip
    to [exp, ICICLE, NEIMS] (exp on top) by swapping axes positions,
    without touching the shared helper other notebooks (QCxMS2 comparison)
    rely on."""
    fig = plot_three_spectra(**kwargs)
    axes = fig.get_axes()
    positions = [ax.get_position() for ax in axes]
    new_order = [2, 0, 1]  # exp (was axes[2]) on top, then icicle, then neims
    for ax, pos in zip([axes[i] for i in new_order], positions):
        ax.set_position(pos)

    bottom_ax = axes[new_order[-1]]
    for ax in axes:
        is_bottom = ax is bottom_ax
        ax.tick_params(
            axis="x", labelbottom=is_bottom, length=3 if is_bottom else 0
        )
        ax.set_xlabel("m/z" if is_bottom else "")
    return fig


def fg_gap_table(fg_summary: pd.DataFrame) -> pd.DataFrame:
    """Functional groups sorted by ICICLE_rate - NEIMS_rate (descending =
    ICICLE-favored first)."""
    out = fg_summary.copy()
    out["gap_icicle_minus_neims"] = out["ICICLE_rate"] - out["NEIMS_rate"]
    return out.sort_values("gap_icicle_minus_neims", ascending=False)


def shortlist_candidates(
    df: pd.DataFrame, functional_group: str, winner: str, n=8
) -> pd.DataFrame:
    """Molecules with this functional group where `winner` is correct
    (top1) and the other model is not — a shortlist to pick a clean
    illustrative example from manually, rather than one auto-picked row."""
    mask = df["functional_groups"].apply(lambda gs: functional_group in gs)
    sub = df[mask].copy()
    if winner == "ICICLE":
        sub = sub[(sub["ICICLE_rate"] == 1.0) & (sub["NEIMS_rate"] == 0.0)]
    else:
        sub = sub[(sub["NEIMS_rate"] == 1.0) & (sub["ICICLE_rate"] == 0.0)]
    cols = [
        c
        for c in ["mol_id", "formula", "mw", "inchikey14"]
        if c in sub.columns
    ]
    return sub[cols].head(n)


def load_icicle_rank_decoys_similarity(seed_dirs: list, mol_id: int) -> dict:
    for d in seed_dirs:
        raw = pd.read_csv(d / "retrieval_with_formula_results.csv")
        rows = raw[raw["spec"] == mol_id]
        if len(rows) == 0:
            continue
        true_row = rows[~rows["is_decoy"]]
        if len(true_row) == 0:
            continue
        r = true_row.iloc[0]
        return {
            "rank_cosine": int(r["rank_cosine_similarity"]),
            "rank_entropy": int(r["rank_entropy_similarity"]),
            "cosine_similarity": float(r["cosine_similarity"]),
            "entropy_similarity": float(r["entropy_similarity"]),
            "n_decoys": int(rows["is_decoy"].sum()),
        }
    return {}


def load_neims_rank_decoys_similarity(
    seed_dirs: list, inchikey14: str
) -> dict:
    for d in seed_dirs:
        raw = pd.read_csv(d / "retrieval_with_formula_results.csv")
        rows = raw[raw["query_inchikey14"] == inchikey14]
        if len(rows) == 0:
            continue
        true_row = rows[rows["is_correct"]]
        if len(true_row) == 0:
            continue
        r = true_row.iloc[true_row["rank_cosine_similarity"].values.argmin()]
        return {
            "rank_cosine": int(r["rank_cosine_similarity"]),
            "rank_entropy": int(r["rank_entropy_similarity"]),
            "cosine_similarity": float(r["cosine_similarity"]),
            "entropy_similarity": float(r["entropy_similarity"]),
            "n_decoys": int(len(rows) - len(true_row)),
        }
    return {}


def load_global_rank_decoys_similarity(
    split: str, model: str, mol_id: int
) -> dict:
    raw = pd.read_csv(
        GLOBAL_RETRIEVAL_DIRS[split][model] / "retrieval_global_per_query.tsv",
        sep="\t",
    )
    row = raw[raw["mol_id"] == mol_id]
    if len(row) == 0:
        return {}
    r = row.iloc[0]
    return {
        "rank_cosine": int(r["rank_autofail_cosine"]),
        "rank_entropy": None,
        "cosine_similarity": None,
        "entropy_similarity": None,
        "n_decoys": int(r["n_candidates"]) - 1,
    }


def load_neims_predicted_spectrum(split: str, mol_id: int) -> np.ndarray:
    for path in NEIMS_PRED_FILES[split]:
        with h5py.File(path, "r") as f:
            key = str(mol_id)
            if key in f:
                return f[key]["predicted_intensities"][:]
    return None


def load_icicle_eval_group(split: str, mol_id: int):
    """`all_evaluation_spectra.hdf5` is keyed by the FULL InChIKey (27
    chars), not InChIKey-14. Returns (mz_bins, predicted, ground_truth),
    all length-750 arrays on the same binning, or (None, None, None)."""
    full_inchikey = metadata_full.loc[
        metadata_full["mol_id"] == mol_id, "inchi_key"
    ].iloc[0]
    for d in SIM_DIRS[split]:
        with h5py.File(d / "all_evaluation_spectra.hdf5", "r") as f:
            if full_inchikey in f:
                grp = f[full_inchikey]
                return (
                    grp["mz_bins"][:],
                    grp["predicted_intensities"][:],
                    grp["ground_truth_intensities"][:],
                )
    return None, None, None


def plot_win_example(
    regime: str,
    mol_id: int,
    functional_group: str,
    winner: str,
    output_dir: Path,
):
    """regime: 'random', 'scaffold', 'global', or 'global_scaffold'. For the
    two global regimes, spectra are read from the corresponding split's
    ICICLE similarity eval + NEIMS predictions (global retrieval itself has
    no per-model spectrum HDF5, only ranks) since ICICLE/NEIMS predicted
    spectra don't depend on which retrieval regime is being illustrated."""
    row = metadata_full[metadata_full["mol_id"] == mol_id].iloc[0]
    inchikey14 = row["inchikey14"]
    is_global = regime.startswith("global")
    global_split = regime.split("_", 1)[1] if "_" in regime else "random"
    spectrum_split = global_split if is_global else regime

    mz, icicle_intensities, exp_intensities = load_icicle_eval_group(
        spectrum_split, mol_id
    )
    neims_intensities = load_neims_predicted_spectrum(spectrum_split, mol_id)
    if (
        icicle_intensities is None
        or exp_intensities is None
        or neims_intensities is None
    ):
        print(f"  Skipping mol_id={mol_id}: missing predicted spectrum")
        return None

    if is_global:
        icicle_stats = load_global_rank_decoys_similarity(
            global_split, "ICICLE", mol_id
        )
        neims_stats = load_global_rank_decoys_similarity(
            global_split, "NEIMS", mol_id
        )
    else:
        icicle_stats = load_icicle_rank_decoys_similarity(
            FORMULA_DIRS[regime]["ICICLE"], mol_id
        )
        neims_stats = load_neims_rank_decoys_similarity(
            FORMULA_DIRS[regime]["NEIMS"], inchikey14
        )

    print(
        f"  [{regime}] {functional_group!r}, {winner}-favored: mol_id={mol_id}, formula={row['formula']}\n"
        f"    ICICLE: {icicle_stats}\n    NEIMS: {neims_stats}"
    )

    def metric_str(stats):
        if not stats or stats.get("entropy_similarity") is None:
            return f"rank {stats.get('rank_cosine')}/{stats.get('n_decoys')} decoys"
        return (
            f"rank {stats['rank_cosine']}/{stats['n_decoys']} decoys, "
            f"cos={stats['cosine_similarity']:.3f}, entr={stats['entropy_similarity']:.3f}"
        )

    fig = plot_stacked_exp_icicle_neims(
        exp_spec=exp_intensities,
        qcxms_spec=neims_intensities,
        icicle_spec=icicle_intensities,
        mz_values=mz,
        smiles=row["standardized_smiles"],
        fade_unmatched=True,
        qcxms_label="NEIMS",
        icicle_color=MODEL_COLORS["ICICLE"],
        qcxms_color=MODEL_COLORS["NEIMS"],
        figsize=(3, 1.5),
    )
    stem = f"stacked_spectrum_{regime}_{winner.lower()}_favored_{functional_group.replace(' ', '_')}"
    save_fig(fig, stem, output_dir)
    print(f"  Saved {stem}.[svg|png]")
    return {
        "regime": regime,
        "functional_group": functional_group,
        "winner": winner,
        "mol_id": mol_id,
        "inchikey14": inchikey14,
        "formula": row["formula"],
        "smiles": row["standardized_smiles"],
        **{f"icicle_{k}": v for k, v in icicle_stats.items()},
        **{f"neims_{k}": v for k, v in neims_stats.items()},
    }

In [ ]:
REGIME_DATA = {
    "random": (random_formula_merged, random_fg_summary),
    "scaffold": (scaffold_formula_merged, scaffold_fg_summary),
    "global": (global_merged, global_fg_summary),
    "global_scaffold": (global_scaffold_merged, global_scaffold_fg_summary),
}

N_FUNCTIONAL_GROUPS = 3  # top-N favored functional groups to search per side
N_PER_GROUP = 2  # candidate molecules to keep per functional group

shortlists = {}
for regime, (merged, fg_summary) in REGIME_DATA.items():
    gap = fg_gap_table(fg_summary[["n", "ICICLE_rate", "NEIMS_rate"]])
    print(
        f"=== {regime.upper()}: functional groups by ICICLE-vs-NEIMS gap ==="
    )
    print(f"Top {N_FUNCTIONAL_GROUPS} ICICLE-favored:")
    display(gap.head(N_FUNCTIONAL_GROUPS))
    print(f"Top {N_FUNCTIONAL_GROUPS} NEIMS-favored:")
    display(gap.tail(N_FUNCTIONAL_GROUPS))

    for winner, fg_candidates in [
        ("ICICLE", gap.head(N_FUNCTIONAL_GROUPS).index),
        ("NEIMS", gap.tail(N_FUNCTIONAL_GROUPS).index[::-1]),
    ]:
        for fg in fg_candidates:
            candidates = shortlist_candidates(
                merged, fg, winner, n=N_PER_GROUP
            )
            if len(candidates) > 0:
                print(f"  Shortlist for {regime}/{winner}-favored/{fg!r}:")
                display(candidates)
                shortlists[(regime, winner, fg)] = candidates

In [ ]:
# Plot every candidate in every shortlist (up to N_PER_GROUP per functional
# group per regime/winner) rather than just the first pick.
example_records = []
for (regime, winner, fg), candidates in shortlists.items():
    for _, cand in candidates.iterrows():
        record = plot_win_example(
            regime, int(cand["mol_id"]), fg, winner, OUTPUT_DIR
        )
        if record is not None:
            example_records.append(record)

example_summary = pd.DataFrame(example_records)
example_summary.to_csv(
    OUTPUT_DIR / "win_loss_examples_rank_decoys.csv", index=False
)
print(f"Plotted {len(example_summary)} examples total")
example_summary

In [ ]:
# Cosine similarity for the global-retrieval examples: the global
# per-query TSVs only carry rank (no formula/decoy set to score similarity
# against), so compute cosine directly from each model's predicted
# spectrum vs. ground truth, reusing icicle.analysis.metrics.cosine_similarity,
# and write it back into example_summary/the saved CSV.
from icicle.analysis.metrics import cosine_similarity

global_mask = example_summary["regime"].isin(["global", "global_scaffold"])
for idx, ex in example_summary[global_mask].iterrows():
    mol_id = int(ex["mol_id"])
    spectrum_split = (
        "scaffold" if ex["regime"] == "global_scaffold" else "random"
    )
    _, icicle_pred, gt = load_icicle_eval_group(spectrum_split, mol_id)
    neims_pred = load_neims_predicted_spectrum(spectrum_split, mol_id)
    icicle_cos = (
        float(cosine_similarity(icicle_pred, gt))
        if icicle_pred is not None
        else None
    )
    neims_cos = (
        float(cosine_similarity(neims_pred, gt))
        if neims_pred is not None
        else None
    )
    example_summary.loc[idx, "icicle_cosine_similarity"] = icicle_cos
    example_summary.loc[idx, "neims_cosine_similarity"] = neims_cos
    print(
        f"[{ex['regime']}] {ex['functional_group']!r} ({ex['winner']}-favored), mol_id={mol_id}: "
        f"ICICLE cosine={icicle_cos:.3f}, NEIMS cosine={neims_cos:.3f}"
    )

example_summary.to_csv(
    OUTPUT_DIR / "win_loss_examples_rank_decoys.csv", index=False
)
example_summary